In [2]:
import math
import numpy as np

Building from the micrograd I created a basic autograd engine that works with tensors.

A large proportion of the structure remains the same or very similar. Basic operations such as addition, multiplication, subtraction, powers and division largely remain the same with the addition of handling broadcasting for grads.

Several features are required for use of tensors rather than using scalars:
- Tensor multiplication (in terms of batch matrix multiplication) must be introduced and hence the grads as well
- Handing broadcasting and in particular how grads are calculated

In addition to the above I introduced other useful operations commonly used in neural networks, which were all straightforward to implement:
- reshaping
- transposing for general tensors (swapping the last 2 axis)
- ReLU
- softmax
- logsoftmax
- max
- log
- sums
- means

We take advantage of the nature of the operations we do to form compact representations of the jacobian. For example in addition, if Y=X+Z, only the (i,j) entry of X and Z affect the (i,j) entry of Y hence we don't need to even think about the other entries (which will have grad 0).

To handle broadcasting we can just think of a mapping from the original to a broadcasted version which is used in the next calculation. We can then use backpropagation with the chain rule to calculate the grad.

In [3]:
import numpy as np

class Tensor:
    def __init__(self,data,prev=(),requires_grad=False):
        self.data=np.asarray(data, dtype=np.float64)
        self.parent=tuple(prev)
        self.requires_grad=requires_grad
        self.grad=np.zeros_like(self.data) if requires_grad else None
        self._backward=lambda: None
        self.shape=np.shape(self.data)
    
    def __add__(self,other):
        if not isinstance(other,Tensor):
            other=Tensor(other)
        out=Tensor(self.data+other.data,(self,other),requires_grad=self.requires_grad or other.requires_grad)
        def _backward():
            if self.requires_grad:
                self.grad+=Tensor.unbroadcast(out.grad,self.shape)
            if other.requires_grad:
                other.grad+=Tensor.unbroadcast(out.grad,other.shape)
        out._backward=_backward
        return out

    def __mul__(self,other):
        if not isinstance(other,Tensor):
            other=Tensor(other)
        out=Tensor(self.data*other.data,(self,other),requires_grad=self.requires_grad or other.requires_grad)
        def _backward():
            if self.requires_grad:
                self.grad+=Tensor.unbroadcast(other.data*out.grad,self.shape)
            if other.requires_grad:
                other.grad+=Tensor.unbroadcast(self.data*out.grad,other.shape)
        out._backward=_backward
        return out

    def __pow__(self,power):
        out=Tensor(self.data**power,(self,),requires_grad=self.requires_grad)
        def _backward():
            if self.requires_grad:
                self.grad+=power*(self.data**(power-1))*out.grad
        out._backward=_backward
        return out
    
    def __truediv__(self,other):
        return self*(other**-1)
    
    def __neg__(self):
        return self*(-1)
    
    def __sub__(self,other):
        if not isinstance(other,Tensor):
            other=Tensor(other)
        return self+-other
    
    def __radd__(self,other):
        if not isinstance(other,Tensor):
            other=Tensor(other)
        return other+self
        
    def __rsub__(self,other):
        if not isinstance(other,Tensor):
            other=Tensor(other)
        return -other+self

    def __rmul__(self,other):
        if not isinstance(other,Tensor):
            other=Tensor(other)
        return other*self
    
    def exp(self):
        out=Tensor(np.exp(self.data),(self,),requires_grad=self.requires_grad)
        def _backward():
            if self.requires_grad:
                self.grad+=out.data*out.grad
        out._backward=_backward
        return out
    
    def backward(self,grad=None):
        topo=[]
        visited=set()
        def topo_build(v):
            if v not in visited:
                visited.add(v)
                for parent in v.parent:
                    topo_build(parent)
                topo.append(v)
        topo_build(self)
        for node in topo:
            if node.requires_grad:
                node.zero_grad()
        if grad is None:
            self.grad=np.ones_like(self.data)
        else:
            self.grad=grad
        for node in reversed(topo):
            node._backward()

    def __rtruediv__(self,other):
        if not isinstance(other,Tensor):
            other=Tensor(other)
        return other/self

    def zero_grad(self):
        if self.requires_grad:
            self.grad=np.zeros_like(self.data)
    
    def sum(self,axis=None,keepdims=False):
        out=Tensor(self.data.sum(axis=axis,keepdims=keepdims),(self,),requires_grad=self.requires_grad)
        def _backward():
            if self.requires_grad:
                grad=out.grad
                if axis is None:
                    grad=np.ones_like(self.data)*grad
                else:
                    if not keepdims:
                        #Adds on additional axis to grad if axis is removed when summing
                        grad=np.expand_dims(grad,axis)
                    #Broadcasts the grad to the shape of self
                    grad=np.broadcast_to(grad,self.data.shape)
                self.grad+=grad
        out._backward=_backward
        return out

    def mean(self,axis=None,keepdims=False):
        if axis is None:
            n=self.data.size
        else:
            n=self.data.shape[axis]
        return self.sum(axis=axis,keepdims=keepdims)/n
    
    @staticmethod
    def unbroadcast(grad,shape):
        #We repeatedly sum over the first axis until we have the same shape as the desired shape 
        #Numpy always adds missing dimensions to the left when broadcasting hence we sum over axis=0
        while len(grad.shape)>len(shape):
            grad=grad.sum(axis=0)
        #We then collapse all axis which are supposed to be dimension 1
        for i,dim in enumerate(shape):
            if dim==1 and grad.shape[i]!=1:
                grad=grad.sum(axis=i,keepdims=True)
        return grad
    
    def reshape(self, shape):
        out=Tensor(self.data.reshape(shape),(self,),requires_grad=self.requires_grad)
        def _backward():
            if self.requires_grad:
                self.grad+=out.grad.reshape(self.shape)
        out._backward=_backward
        return out
    
    def transpose(self,axis1=-2,axis2=-1):
        out=Tensor(np.swapaxes(self.data,axis1,axis2),(self,),requires_grad=self.requires_grad)
        def _backward():
            if self.requires_grad:
                self.grad+=np.swapaxes(out.grad,axis1,axis2)
        out._backward=_backward
        return out
    
    def __matmul__(self,other):
        if not isinstance(other,Tensor):
            other=Tensor(other)
        out=Tensor(self.data@other.data,(self,other),requires_grad=self.requires_grad or other.requires_grad)
        def _backward():
            if self.requires_grad:
                self.grad+=Tensor.unbroadcast(out.grad@np.swapaxes(other.data,-2,-1),self.shape)
            if other.requires_grad:
                other.grad+=Tensor.unbroadcast(np.swapaxes(self.data,-2,-1)@out.grad,other.shape)
        out._backward=_backward
        return out
    
    def log(self):
        out=Tensor(np.log(self.data),(self,),requires_grad=self.requires_grad)
        def _backward():
            if self.requires_grad:
                self.grad+=out.grad/self.data
        out._backward=_backward
        return out
    
    def ReLU(self):
        #self.data>0 is a compact representation of the jacobian
        out=Tensor(np.maximum(0,self.data),(self,),requires_grad=self.requires_grad)
        def _backward():
            if self.requires_grad:
                self.grad+=(self.data>0)*out.grad
        out._backward=_backward
        return out
    
    def max(self,axis=None,keepdims=False):
        #In max, mask is the compact representation of the jacobian. 
        out=Tensor(np.max(self.data, axis=axis, keepdims=keepdims),(self,),requires_grad=self.requires_grad)
        def _backward():
            if self.requires_grad:
                grad=out.grad
                if axis is None:
                    mask=(self.data==np.max(self.data))
                    self.grad+=mask*grad
                else:
                    if not keepdims:
                        grad=np.expand_dims(grad,axis)
                    mask=(self.data==np.max(self.data,axis=axis,keepdims=True))
                    self.grad+=mask*grad
        out._backward=_backward
        return out
    
    def softmax(self,axis=-1):
        shifted=self-self.max(axis=axis,keepdims=True)
        exp=shifted.exp()
        return exp/exp.sum(axis=axis,keepdims=True)
    
    def __rmatmul__(self,other):
        if not isinstance(other,Tensor):
            other=Tensor(other)
        return other@self
    
    def log_softmax(self,axis=-1):
        shifted=self-self.max(axis=axis,keepdims=True)
        return shifted-shifted.exp().sum(axis=axis,keepdims=True).log()

All operations are checked below in terms of getting the correct output and correct grad on backward propagation. The grad is found through the finite differences method. This was created using gen-AI.

In [3]:
def test_autograd():
    def numerical_grad(f, x, eps=1e-5):
        grad = np.zeros_like(x)
        for idx in np.ndindex(x.shape):
            old = x[idx]
            x[idx] = old + eps
            plus = f(x)
            x[idx] = old - eps
            minus = f(x)
            x[idx] = old
            grad[idx] = (plus - minus) / (2 * eps)
        return grad
    def check(name, func, x):
        t = Tensor(x.copy(), requires_grad=True)
        loss = func(t)
        loss.backward()
        analytic = t.grad.copy()
        numeric = numerical_grad(lambda z: func(Tensor(z)).data,x.copy())
        passed = np.allclose(analytic,numeric,atol=1e-5)
        print(f"{name}: {'PASS' if passed else 'FAIL'}")
        if not passed:
            print("analytic:")
            print(analytic)
            print("numeric:")
            print(numeric)
    # addition
    check(
        "add",
        lambda x: (x+x).sum(),
        np.random.randn(3,3)
    )
    # multiplication
    check(
        "multiply",
        lambda x: (x*x).sum(),
        np.random.randn(3,3)
    )
    # power
    check(
        "power",
        lambda x: (x**3).sum(),
        np.random.randn(3,3)
    )
    # exp
    check(
        "exp",
        lambda x: x.exp().sum(),
        np.random.randn(3,3)
    )
    # log (positive values)
    check(
        "log",
        lambda x: x.log().sum(),
        np.random.rand(3,3)+1
    )
    # relu
    check(
        "relu",
        lambda x: x.ReLU().sum(),
        np.array([
            [-2.,1.,3.],
            [4.,-1.,5.]
        ])
    )
    # broadcasting
    check(
        "broadcast add",
        lambda x: (x + Tensor(np.ones(3))).sum(),
        np.random.randn(2,3)
    )
    # sum
    check(
        "sum",
        lambda x: x.sum(),
        np.random.randn(3,3)
    )
    # mean
    check(
        "mean",
        lambda x: x.mean(),
        np.random.randn(3,3)
    )
    # reshape
    check(
        "reshape",
        lambda x: x.reshape((9,)).sum(),
        np.random.randn(3,3)
    )
    # transpose
    check(
        "transpose",
        lambda x: x.transpose().sum(),
        np.random.randn(3,4)
    )
    # matrix multiplication
    W = np.random.randn(4,2)

    check(
        "matmul",
        lambda x: (x @ Tensor(W)).sum(),
        np.random.randn(3,4)
    )
    # softmax
    check(
        "softmax",
        lambda x: (x.softmax()*x).sum(),
        np.random.randn(2,5)
    )
    print("\nFinished testing")
test_autograd()

add: PASS
multiply: PASS
power: PASS
exp: PASS
log: PASS
relu: PASS
broadcast add: PASS
sum: PASS
mean: PASS
reshape: PASS
transpose: PASS
matmul: PASS
softmax: PASS

Finished testing
